# Day 4 - Gate L2: Lee Three-Asset Brownian-Bridge Exit Probabilities

## tl;dr

**Gate L2: LIMITED / FAIL.**

- Appendix C selected estimate: **0.165739**
  versus the paper target 0.1631.
- Table 1 reproduced RMS relative error: **BB 0.0071**
  versus **direct MC 0.0552**.
- Out-of-sample logistic \(q_3\) RMSE versus the nested extrapolated
  reference: **0.003706**, against the pre-registered 0.0025
  probability-point target.
- Probability-bound violations: **q3=0**,
  **p3=0**.
- Failed hard criteria: **Approximation bias below target sampling RMSE**.

This notebook is an independent implementation of Lee, Ha, Kong, and Lee
(2024). If Gate L2 is limited/failed, the saved nested lookup is the
required fallback and no exact-reproduction claim is made.

## Context & Methods

The source is Lee, H., Ha, H., Kong, B., and Lee, M. (2024),
*Valuing three-asset barrier options and autocallable products via exit
probabilities of Brownian bridge*, **The North American Journal of
Economics and Finance 73**, 102174,
[doi:10.1016/j.najef.2024.102174](https://doi.org/10.1016/j.najef.2024.102174).

For an upper barrier \(b_i\), endpoint \(x_i\le b_i\), volatility
\(\sigma_i\), and interval length \(T\), the marginal bridge exit is

\[
g_i(x_i;b_i)=
\exp\left[-\frac{2b_i(b_i-x_i)}{\sigma_i^2T}\right].
\]

The bivariate co-exit \(h_{ij}\) is evaluated from Proposition 2.1 with
the modified-Bessel series. For the trivariate co-exit, each valid
decomposition is

\[
q_3(\mathbf x)
=h_{ij}(\mathbf x)\,
\Pr(M_k>b_k\mid M_i>b_i,M_j>b_j,\mathbf X(T)=\mathbf x),
\]

where \(k\notin\{i,j\}\). The frozen inference rule chooses the smallest
of \(h_{12},h_{13},h_{23}\) and its corresponding conditional model.
This guarantees \(0\le\widehat q_3\le\min h_{ij}\). Joint non-exit is

\[
\widehat p_3
=1-\sum_i g_i+\sum_{i<j}h_{ij}-\widehat q_3.
\]

### Key Assumptions

- The implementation is independent; it does not claim source-code replication.
- The paper does not disclose its training seed, training path grid,
  solver, regularisation, feature scaling, or extrapolation policy.
- Baseline Appendix C training uses seed `20260731`, 100,000 candidate
  paths, 25 time steps, unpenalised logistic maximum likelihood,
  L-BFGS-B, and standardised endpoint features.
- A 0.005 absolute tolerance is fixed for the reported Appendix C target
  \(0.1631\).
- The main-experiment target sampling RMSE is fixed at 0.0025 probability
  points before viewing the robustness results.
- Nested references use common paths at 250 and 1,000 time steps and
  extrapolate linearly in \(1/\sqrt{m}\). Their Monte Carlo standard
  errors are reported separately.
- Results outside the training feature domain are not silently
  extrapolated; they require a validated lookup/interpolation entry or
  a fresh nested reference.

In [ ]:
from pathlib import Path
import hashlib
import json
import math
import os
import platform
import sys
import tempfile
import time
import warnings

import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.special import expit, ive

os.environ.setdefault(
    "MPLCONFIGDIR",
    str(Path(tempfile.gettempdir()) / "day4_lee_mpl_cache"),
)
import matplotlib.pyplot as plt

pd.set_option("display.precision", 8)
plt.style.use("seaborn-v0_8-whitegrid")


def find_project_root():
    candidates = []
    override = os.environ.get("AP_PROJECT_ROOT")
    if override:
        candidates.append(Path(override).expanduser())
    candidates.extend([Path.cwd(), *Path.cwd().parents])
    notebook_dir = Path(
        globals().get("__file__", Path.cwd())
    ).resolve().parent
    candidates.extend([notebook_dir, *notebook_dir.parents])
    for candidate in candidates:
        if (
            candidate / "config" / "core_project_config.json"
        ).is_file():
            return candidate.resolve()
    raise FileNotFoundError(
        "Could not locate config/core_project_config.json. "
        "Run from the project directory or set AP_PROJECT_ROOT."
    )


PROJECT_DIR = find_project_root()
OUTPUT_DIR = PROJECT_DIR / "outputs" / "day4_lee_exit_probability"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SOURCE_PDF = PROJECT_DIR / "references" / "lee_2024.pdf"
PLAN_FILE = PROJECT_DIR / "docs" / "research-plan.md"

BASE_SEED = 20260731
APPENDIX_TARGET = 0.1631
APPENDIX_TOLERANCE = 0.005
TARGET_SAMPLING_RMSE = 0.0025
BESSEL_TERMS = 12

APPENDIX_MU = np.full(3, 0.03)
APPENDIX_SIGMA = np.full(3, 0.20)
APPENDIX_CORRELATION = np.full((3, 3), 0.40)
np.fill_diagonal(APPENDIX_CORRELATION, 1.0)
APPENDIX_BARRIER = np.full(3, 0.10)
APPENDIX_T = 0.50
APPENDIX_X = np.array([0.01, 0.02, 0.03])

PAIR_SPECS = [
    ("h12", 0, 1, 2),
    ("h13", 0, 2, 1),
    ("h23", 1, 2, 0),
]

print(f"Project root: {PROJECT_DIR}")
print(f"Paper available: {SOURCE_PDF.is_file()} ({SOURCE_PDF})")
print(f"Plan available: {PLAN_FILE.is_file()} ({PLAN_FILE})")
print(f"Output directory: {OUTPUT_DIR}")

## Data

### 1. Implement and audit \(g_i\) and \(h_{ij}\)

Angles in Proposition 2.1 are measured clockwise from the negative
\(z_1\)-axis. The implementation uses the exponentially scaled modified
Bessel function `scipy.special.ive` so that
\(I_\nu(a)=e^a\,\mathrm{ive}(\nu,a)\) for \(a\ge0\).

In [ ]:
def validate_inputs(sigma, correlation):
    sigma = np.asarray(sigma, dtype=float)
    correlation = np.asarray(correlation, dtype=float)
    if sigma.shape != (3,) or np.any(sigma <= 0):
        raise ValueError("sigma must contain three positive values")
    if correlation.shape != (3, 3):
        raise ValueError("correlation must be 3 by 3")
    if not np.allclose(correlation, correlation.T, atol=1e-12):
        raise ValueError("correlation must be symmetric")
    if not np.allclose(np.diag(correlation), 1.0, atol=1e-12):
        raise ValueError("correlation diagonal must equal one")
    eigenvalues = np.linalg.eigvalsh(correlation)
    if eigenvalues.min() <= 1e-10:
        raise ValueError(
            f"correlation is not positive definite: {eigenvalues}"
        )
    return sigma, correlation


def marginal_exit(x, barrier, sigma, maturity):
    x, barrier, sigma = np.broadcast_arrays(
        np.asarray(x, dtype=float),
        np.asarray(barrier, dtype=float),
        np.asarray(sigma, dtype=float),
    )
    if maturity <= 0 or np.any(sigma <= 0) or np.any(barrier < 0):
        raise ValueError(
            "maturity and sigma must be positive; barrier non-negative"
        )
    exponent = -2.0 * barrier * (barrier - x) / (
        sigma * sigma * maturity
    )
    result = np.where(x >= barrier, 1.0, np.exp(exponent))
    result = np.clip(result, 0.0, 1.0)
    return float(result) if result.ndim == 0 else result


def bivariate_coexit(
    x_i,
    x_j,
    b_i,
    b_j,
    sigma_i,
    sigma_j,
    rho,
    maturity,
    n_terms=BESSEL_TERMS,
    clip_probability=True,
):
    if not (-1.0 < rho < 1.0):
        raise ValueError("rho must lie strictly between -1 and 1")
    if maturity <= 0 or sigma_i <= 0 or sigma_j <= 0:
        raise ValueError("maturity and volatilities must be positive")
    if b_i < 0 or b_j < 0:
        raise ValueError("upper barriers must be non-negative")

    x_i, x_j = np.broadcast_arrays(
        np.asarray(x_i, dtype=float),
        np.asarray(x_j, dtype=float),
    )
    scalar = x_i.ndim == 0
    x_i = np.atleast_1d(x_i)
    x_j = np.atleast_1d(x_j)

    g_i = marginal_exit(x_i, b_i, sigma_i, maturity)
    g_j = marginal_exit(x_j, b_j, sigma_j, maturity)
    result = np.empty_like(x_i, dtype=float)

    both_above = (x_i >= b_i) & (x_j >= b_j)
    only_i_above = (x_i >= b_i) & (x_j < b_j)
    only_j_above = (x_i < b_i) & (x_j >= b_j)
    both_below = (x_i < b_i) & (x_j < b_j)

    result[both_above] = 1.0
    result[only_i_above] = g_j[only_i_above]
    result[only_j_above] = g_i[only_j_above]

    if np.any(both_below):
        xi = x_i[both_below]
        xj = x_j[both_below]
        root = math.sqrt(1.0 - rho * rho)
        beta = math.acos(-rho)

        z1 = (
            (xi - b_i) / sigma_i
            - rho * (xj - b_j) / sigma_j
        ) / root
        z2 = (xj - b_j) / sigma_j
        z10 = (-b_i / sigma_i + rho * b_j / sigma_j) / root
        z20 = -b_j / sigma_j

        d = np.hypot(z1, z2)
        d0 = math.hypot(z10, z20)
        theta = np.arctan2(-z2, -z1)
        theta0 = math.atan2(-z20, -z10)
        theta = np.clip(theta, 0.0, beta)
        theta0 = float(np.clip(theta0, 0.0, beta))

        argument = d * d0 / maturity
        series = np.zeros_like(argument)
        for term_index in range(1, int(n_terms) + 1):
            order = term_index * math.pi / beta
            angular = (
                math.sin(term_index * math.pi * theta0 / beta)
                * np.sin(term_index * math.pi * theta / beta)
            )
            series += angular * ive(order, argument)

        exponent = (
            argument - (z1 * z10 + z2 * z20) / maturity
        )
        with np.errstate(over="ignore", invalid="ignore"):
            joint_nonexit = (
                4.0
                * math.pi
                / beta
                * np.exp(exponent)
                * series
            )
        raw = (
            g_i[both_below]
            + g_j[both_below]
            + joint_nonexit
            - 1.0
        )
        result[both_below] = raw

    if clip_probability:
        result = np.clip(result, 0.0, np.minimum(g_i, g_j))
    if np.any(~np.isfinite(result)):
        raise FloatingPointError(
            "Non-finite bivariate probability; inspect parameter domain"
        )
    return float(result[0]) if scalar else result


def pair_probabilities(
    endpoints,
    barriers,
    sigma,
    correlation,
    maturity,
    n_terms=BESSEL_TERMS,
):
    endpoints = np.atleast_2d(np.asarray(endpoints, dtype=float))
    h = np.empty((len(endpoints), 3))
    for pair_column, (_, i, j, _) in enumerate(PAIR_SPECS):
        h[:, pair_column] = bivariate_coexit(
            endpoints[:, i],
            endpoints[:, j],
            barriers[i],
            barriers[j],
            sigma[i],
            sigma[j],
            correlation[i, j],
            maturity,
            n_terms=n_terms,
        )
    return h


paper_h = np.array([0.2375, 0.2565, 0.2790])
truncation_rows = []
for n_terms in [2, 3, 4, 5, 8, 12, 20, 40]:
    values = pair_probabilities(
        APPENDIX_X,
        APPENDIX_BARRIER,
        APPENDIX_SIGMA,
        APPENDIX_CORRELATION,
        APPENDIX_T,
        n_terms=n_terms,
    )[0]
    truncation_rows.append(
        {
            "Bessel terms": n_terms,
            "h12": values[0],
            "h13": values[1],
            "h23": values[2],
            "max abs change vs 40": np.nan,
        }
    )

bessel_truncation = pd.DataFrame(truncation_rows)
reference_40 = bessel_truncation.iloc[-1][
    ["h12", "h13", "h23"]
].to_numpy(dtype=float)
bessel_truncation["max abs change vs 40"] = (
    bessel_truncation[["h12", "h13", "h23"]]
    .sub(reference_40)
    .abs()
    .max(axis=1)
)
analytic_h = bessel_truncation.loc[
    bessel_truncation["Bessel terms"] == BESSEL_TERMS,
    ["h12", "h13", "h23"],
].iloc[0].to_numpy(dtype=float)

display(bessel_truncation)
print("Analytic h:", analytic_h)
print("Paper Table C.9 rounded h:", paper_h)
print("Max rounded-value discrepancy:", np.max(np.abs(analytic_h - paper_h)))

### 2. Generate candidate paths and fit the three logistic decompositions

The model for `h12` is trained only on paths where assets 1 and 2
exit, with asset 3 exit as the response. The other two models are
defined analogously. This explicit pair-to-target mapping resolves the
otherwise ambiguous row labels in Table C.9.

In [ ]:
def diffusion_cholesky(sigma, correlation):
    sigma, correlation = validate_inputs(sigma, correlation)
    return np.diag(sigma) @ np.linalg.cholesky(correlation)


def simulate_training_paths(
    n_candidates,
    max_steps,
    record_steps,
    mu,
    sigma,
    correlation,
    barriers,
    maturity,
    seed,
    batch_size=5000,
):
    record_steps = sorted(set(int(value) for value in record_steps))
    if any(max_steps % value != 0 for value in record_steps):
        raise ValueError(
            "Every recorded grid must divide max_steps exactly"
        )
    mu = np.asarray(mu, dtype=float)
    barriers = np.asarray(barriers, dtype=float)
    cholesky = diffusion_cholesky(sigma, correlation)
    rng = np.random.default_rng(seed)
    dt = maturity / max_steps
    root_dt = math.sqrt(dt)

    endpoints = np.empty((n_candidates, 3))
    exits_by_grid = {
        grid: np.empty((n_candidates, 3), dtype=bool)
        for grid in record_steps
    }
    strides = {grid: max_steps // grid for grid in record_steps}

    for start in range(0, n_candidates, batch_size):
        stop = min(start + batch_size, n_candidates)
        batch = stop - start
        state = np.zeros((batch, 3))
        maxima = {
            grid: np.zeros((batch, 3)) for grid in record_steps
        }
        for step in range(1, max_steps + 1):
            normals = rng.standard_normal((batch, 3))
            state += mu * dt + normals @ cholesky.T * root_dt
            for grid in record_steps:
                if step % strides[grid] == 0:
                    maxima[grid] = np.maximum(
                        maxima[grid], state
                    )
        endpoints[start:stop] = state
        for grid in record_steps:
            exits_by_grid[grid][start:stop] = (
                maxima[grid] > barriers
            )
    return endpoints, exits_by_grid


def fit_logistic(features, response, preprocessing="standardised"):
    features = np.asarray(features, dtype=float)
    response = np.asarray(response, dtype=float)
    if len(features) < 50 or response.min() == response.max():
        raise ValueError(
            "Insufficient effective training data or a single response class"
        )
    if preprocessing == "standardised":
        center = features.mean(axis=0)
        scale = features.std(axis=0, ddof=0)
        scale = np.where(scale > 1e-12, scale, 1.0)
    elif preprocessing == "raw":
        center = np.zeros(features.shape[1])
        scale = np.ones(features.shape[1])
    else:
        raise ValueError("preprocessing must be raw or standardised")
    design = (features - center) / scale

    def objective(weights):
        linear = weights[0] + design @ weights[1:]
        probabilities = expit(linear)
        loss = np.logaddexp(0.0, linear).sum() - response @ linear
        gradient = np.r_[
            np.sum(probabilities - response),
            design.T @ (probabilities - response),
        ]
        return loss, gradient

    result = minimize(
        objective,
        np.zeros(features.shape[1] + 1),
        jac=True,
        method="L-BFGS-B",
        options={"maxiter": 1000, "ftol": 1e-12, "gtol": 1e-8},
    )
    raw_coefficients = result.x[1:] / scale
    raw_intercept = result.x[0] - center @ raw_coefficients
    return {
        "intercept": float(raw_intercept),
        "coefficients": raw_coefficients,
        "standard_intercept": float(result.x[0]),
        "standard_coefficients": result.x[1:],
        "center": center,
        "scale": scale,
        "preprocessing": preprocessing,
        "solver": "scipy.optimize.minimize L-BFGS-B",
        "regularisation": "none",
        "converged": bool(result.success),
        "iterations": int(result.nit),
        "effective_n": int(len(response)),
        "positive_rate": float(response.mean()),
        "message": str(result.message),
    }


def predict_logistic(model, endpoints):
    endpoints = np.atleast_2d(np.asarray(endpoints, dtype=float))
    linear = model["intercept"] + endpoints @ model["coefficients"]
    return expit(linear)


def fit_three_models(
    endpoints,
    exits,
    preprocessing="standardised",
    candidate_limit=None,
):
    if candidate_limit is not None:
        endpoints = endpoints[:candidate_limit]
        exits = exits[:candidate_limit]
    models = {}
    for pair_name, i, j, target in PAIR_SPECS:
        selected = exits[:, i] & exits[:, j]
        models[pair_name] = fit_logistic(
            endpoints[selected],
            exits[selected, target].astype(float),
            preprocessing=preprocessing,
        )
        models[pair_name]["conditioning_pair"] = (i, j)
        models[pair_name]["target_asset"] = target
        models[pair_name]["candidate_n"] = int(len(endpoints))
    return models


def trivariate_components(
    endpoints,
    barriers,
    sigma,
    correlation,
    maturity,
    models,
    n_terms=BESSEL_TERMS,
):
    endpoints = np.atleast_2d(np.asarray(endpoints, dtype=float))
    barriers = np.asarray(barriers, dtype=float)
    sigma = np.asarray(sigma, dtype=float)
    below = np.all(endpoints < barriers, axis=1)

    g = np.column_stack(
        [
            marginal_exit(
                endpoints[:, index],
                barriers[index],
                sigma[index],
                maturity,
            )
            for index in range(3)
        ]
    )
    h = pair_probabilities(
        endpoints,
        barriers,
        sigma,
        correlation,
        maturity,
        n_terms=n_terms,
    )
    conditional = np.column_stack(
        [
            predict_logistic(models[pair_name], endpoints)
            for pair_name, *_ in PAIR_SPECS
        ]
    )
    q_candidates = h * conditional
    selected_pair_index = np.argmin(h, axis=1)
    row_index = np.arange(len(endpoints))
    q_selected = q_candidates[row_index, selected_pair_index]
    p_raw = 1.0 - g.sum(axis=1) + h.sum(axis=1) - q_selected
    p_raw = np.where(below, p_raw, 0.0)
    return {
        "g": g,
        "h": h,
        "conditional": conditional,
        "q_candidates": q_candidates,
        "selected_pair_index": selected_pair_index,
        "q_selected": q_selected,
        "p_raw": p_raw,
        "below": below,
    }


training_started = time.perf_counter()
training_endpoints, training_exits_by_grid = simulate_training_paths(
    n_candidates=100_000,
    max_steps=1000,
    record_steps=[25, 100, 1000],
    mu=APPENDIX_MU,
    sigma=APPENDIX_SIGMA,
    correlation=APPENDIX_CORRELATION,
    barriers=APPENDIX_BARRIER,
    maturity=APPENDIX_T,
    seed=BASE_SEED,
)
baseline_models = fit_three_models(
    training_endpoints,
    training_exits_by_grid[25],
    preprocessing="standardised",
)
training_seconds = time.perf_counter() - training_started
print(f"Baseline training bundle completed in {training_seconds:.2f}s")

## Results

### 3. Appendix C reproduction and the three equivalent decompositions

In [ ]:
appendix_components = trivariate_components(
    APPENDIX_X,
    APPENDIX_BARRIER,
    APPENDIX_SIGMA,
    APPENDIX_CORRELATION,
    APPENDIX_T,
    baseline_models,
)

model_rows = []
for column, (pair_name, i, j, target) in enumerate(PAIR_SPECS):
    model = baseline_models[pair_name]
    coefficients = model["coefficients"]
    model_rows.append(
        {
            "decomposition": f"{pair_name} x L(target {target + 1})",
            "conditioning pair": f"{i + 1},{j + 1}",
            "target asset": target + 1,
            "candidate n": model["candidate_n"],
            "effective n": model["effective_n"],
            "intercept": model["intercept"],
            "coef x1": coefficients[0],
            "coef x2": coefficients[1],
            "coef x3": coefficients[2],
            "L(x)": appendix_components["conditional"][0, column],
            "h_ij": appendix_components["h"][0, column],
            "q3 candidate": appendix_components["q_candidates"][
                0, column
            ],
            "selected smallest h": column
            == appendix_components["selected_pair_index"][0],
        }
    )

appendix_c_reproduction = pd.DataFrame(model_rows)
selected_appendix_q = float(
    appendix_components["q_selected"][0]
)
appendix_error = selected_appendix_q - APPENDIX_TARGET
display(appendix_c_reproduction)
print(
    f"Selected q3={selected_appendix_q:.6f}; "
    f"paper target={APPENDIX_TARGET:.4f}; "
    f"error={appendix_error:+.6f}"
)
print(
    "Raw p3 at Appendix endpoint:",
    float(appendix_components["p_raw"][0]),
)

### 4. Independent high-grid Brownian-bridge reference

The nested reference conditions directly on \(\mathbf X(T)=\mathbf x\).
It records the same bridge at 250 and 1,000 grid points and uses the
standard \(m^{-1/2}\) barrier-discretisation extrapolation. The
extrapolated path variable is retained when computing Monte Carlo
standard errors, so common-path dependence is not ignored.

In [ ]:
def conditional_bridge_reference(
    endpoint,
    barriers,
    sigma,
    correlation,
    maturity,
    n_paths,
    fine_steps=1000,
    coarse_steps=250,
    seed=BASE_SEED + 1000,
    batch_size=5000,
):
    if fine_steps % coarse_steps != 0:
        raise ValueError("coarse_steps must divide fine_steps")
    endpoint = np.asarray(endpoint, dtype=float)
    barriers = np.asarray(barriers, dtype=float)
    cholesky = diffusion_cholesky(sigma, correlation)
    rng = np.random.default_rng(seed)
    stride = fine_steps // coarse_steps
    ratio = math.sqrt(fine_steps / coarse_steps)
    event_names = [
        "g1",
        "g2",
        "g3",
        "h12",
        "h13",
        "h23",
        "q123",
    ]
    coarse_sum = np.zeros(7)
    fine_sum = np.zeros(7)
    extrapolated_sum = np.zeros(7)
    extrapolated_sumsq = np.zeros(7)

    for start in range(0, n_paths, batch_size):
        batch = min(batch_size, n_paths - start)
        state = np.zeros((batch, 3))
        coarse_maximum = np.zeros((batch, 3))
        fine_maximum = np.zeros((batch, 3))

        for step in range(fine_steps):
            time_now = step * maturity / fine_steps
            dt = maturity / fine_steps
            remaining = maturity - time_now
            if step == fine_steps - 1:
                state = np.broadcast_to(
                    endpoint, (batch, 3)
                ).copy()
            else:
                conditional_variance = (
                    dt * (remaining - dt) / remaining
                )
                state += (
                    (endpoint - state) * dt / remaining
                    + rng.standard_normal((batch, 3))
                    @ cholesky.T
                    * math.sqrt(conditional_variance)
                )
            fine_maximum = np.maximum(fine_maximum, state)
            if (step + 1) % stride == 0:
                coarse_maximum = np.maximum(
                    coarse_maximum, state
                )

        coarse_exit = coarse_maximum > barriers
        fine_exit = fine_maximum > barriers

        def event_matrix(exit_flags):
            return np.column_stack(
                [
                    exit_flags[:, 0],
                    exit_flags[:, 1],
                    exit_flags[:, 2],
                    exit_flags[:, 0] & exit_flags[:, 1],
                    exit_flags[:, 0] & exit_flags[:, 2],
                    exit_flags[:, 1] & exit_flags[:, 2],
                    np.all(exit_flags, axis=1),
                ]
            ).astype(float)

        coarse_events = event_matrix(coarse_exit)
        fine_events = event_matrix(fine_exit)
        extrapolated = (
            ratio * fine_events - coarse_events
        ) / (ratio - 1.0)
        coarse_sum += coarse_events.sum(axis=0)
        fine_sum += fine_events.sum(axis=0)
        extrapolated_sum += extrapolated.sum(axis=0)
        extrapolated_sumsq += (extrapolated**2).sum(axis=0)

    coarse_probability = coarse_sum / n_paths
    fine_probability = fine_sum / n_paths
    extrapolated_probability = extrapolated_sum / n_paths
    sample_variance = (
        extrapolated_sumsq
        - n_paths * extrapolated_probability**2
    ) / (n_paths - 1)
    extrapolated_se = np.sqrt(
        np.maximum(sample_variance, 0.0) / n_paths
    )
    return pd.DataFrame(
        {
            "event": event_names,
            f"grid {coarse_steps}": coarse_probability,
            f"grid {fine_steps}": fine_probability,
            "extrapolated reference": extrapolated_probability,
            "reference SE": extrapolated_se,
            "n paths": n_paths,
            "seed": seed,
        }
    )


reference_started = time.perf_counter()
appendix_reference = conditional_bridge_reference(
    APPENDIX_X,
    APPENDIX_BARRIER,
    APPENDIX_SIGMA,
    APPENDIX_CORRELATION,
    APPENDIX_T,
    n_paths=120_000,
    seed=BASE_SEED + 1000,
)
reference_seconds = time.perf_counter() - reference_started

analytic_reference = np.r_[
    [
        marginal_exit(
            APPENDIX_X[index],
            APPENDIX_BARRIER[index],
            APPENDIX_SIGMA[index],
            APPENDIX_T,
        )
        for index in range(3)
    ],
    analytic_h,
    selected_appendix_q,
]
appendix_reference["analytic / logistic"] = analytic_reference
appendix_reference["difference"] = (
    appendix_reference["analytic / logistic"]
    - appendix_reference["extrapolated reference"]
)
appendix_reference["z score"] = (
    appendix_reference["difference"]
    / appendix_reference["reference SE"]
)
display(appendix_reference)
print(
    f"Conditional bridge reference completed in "
    f"{reference_seconds:.2f}s"
)

### 5. Training-size, time-grid, preprocessing, and seed robustness

In [ ]:
def appendix_model_result(models, label):
    result = trivariate_components(
        APPENDIX_X,
        APPENDIX_BARRIER,
        APPENDIX_SIGMA,
        APPENDIX_CORRELATION,
        APPENDIX_T,
        models,
    )
    row = {
        "setting": label,
        "selected q3": float(result["q_selected"][0]),
        "raw p3": float(result["p_raw"][0]),
        "selected pair": PAIR_SPECS[
            int(result["selected_pair_index"][0])
        ][0],
    }
    for column, (pair_name, *_rest) in enumerate(PAIR_SPECS):
        row[f"{pair_name} decomposition"] = float(
            result["q_candidates"][0, column]
        )
        row[f"{pair_name} effective n"] = models[pair_name][
            "effective_n"
        ]
    return row


training_size_rows = []
for candidate_n in [10_000, 40_000, 100_000]:
    models = fit_three_models(
        training_endpoints,
        training_exits_by_grid[25],
        preprocessing="standardised",
        candidate_limit=candidate_n,
    )
    training_size_rows.append(
        {
            "candidate n": candidate_n,
            **appendix_model_result(
                models, f"candidate n={candidate_n}"
            ),
        }
    )
training_size_robustness = pd.DataFrame(training_size_rows)

training_grid_rows = []
for grid in [25, 100, 1000]:
    models = fit_three_models(
        training_endpoints,
        training_exits_by_grid[grid],
        preprocessing="standardised",
    )
    training_grid_rows.append(
        {
            "training steps": grid,
            **appendix_model_result(
                models, f"training steps={grid}"
            ),
        }
    )
training_grid_robustness = pd.DataFrame(training_grid_rows)

raw_models = fit_three_models(
    training_endpoints,
    training_exits_by_grid[25],
    preprocessing="raw",
)
preprocessing_robustness = pd.DataFrame(
    [
        {
            "preprocessing": "standardised",
            **appendix_model_result(
                baseline_models, "standardised"
            ),
        },
        {
            "preprocessing": "raw",
            **appendix_model_result(raw_models, "raw"),
        },
    ]
)

seed_rows = []
for training_seed in [20260731, 20260732, 20260733, 20260734]:
    seed_endpoints, seed_exits = simulate_training_paths(
        n_candidates=40_000,
        max_steps=25,
        record_steps=[25],
        mu=APPENDIX_MU,
        sigma=APPENDIX_SIGMA,
        correlation=APPENDIX_CORRELATION,
        barriers=APPENDIX_BARRIER,
        maturity=APPENDIX_T,
        seed=training_seed,
    )
    seed_models = fit_three_models(
        seed_endpoints,
        seed_exits[25],
        preprocessing="standardised",
    )
    seed_rows.append(
        {
            "training seed": training_seed,
            **appendix_model_result(
                seed_models, f"seed={training_seed}"
            ),
        }
    )
training_seed_robustness = pd.DataFrame(seed_rows)

display(training_size_robustness)
display(training_grid_robustness)
display(preprocessing_robustness)
display(training_seed_robustness)
print(
    "Independent-seed q3 SD:",
    training_seed_robustness["selected q3"].std(ddof=1),
)

### 6. Correlation, barrier-distance, and asymmetric-parameter robustness

In [ ]:
def train_and_evaluate_scenario(
    scenario,
    mu,
    sigma,
    correlation,
    barriers,
    endpoint,
    seed,
    candidates=40_000,
    training_steps=25,
):
    endpoints, exits_by_grid = simulate_training_paths(
        n_candidates=candidates,
        max_steps=training_steps,
        record_steps=[training_steps],
        mu=mu,
        sigma=sigma,
        correlation=correlation,
        barriers=barriers,
        maturity=APPENDIX_T,
        seed=seed,
    )
    models = fit_three_models(
        endpoints,
        exits_by_grid[training_steps],
        preprocessing="standardised",
    )
    result = trivariate_components(
        endpoint,
        barriers,
        sigma,
        correlation,
        APPENDIX_T,
        models,
    )
    return {
        "scenario": scenario,
        "sigma": json.dumps(np.asarray(sigma).round(4).tolist()),
        "barriers": json.dumps(
            np.asarray(barriers).round(4).tolist()
        ),
        "correlation": json.dumps(
            np.asarray(correlation).round(4).tolist()
        ),
        "selected pair": PAIR_SPECS[
            int(result["selected_pair_index"][0])
        ][0],
        "q3": float(result["q_selected"][0]),
        "p3": float(result["p_raw"][0]),
        "decomposition range": float(
            np.ptp(result["q_candidates"][0])
        ),
    }


scenario_rows = []
scenario_seed = BASE_SEED + 2000
for rho in [0.10, 0.40, 0.70]:
    correlation = np.full((3, 3), rho)
    np.fill_diagonal(correlation, 1.0)
    scenario_rows.append(
        train_and_evaluate_scenario(
            f"equicorrelation {rho:.1f}",
            APPENDIX_MU,
            APPENDIX_SIGMA,
            correlation,
            APPENDIX_BARRIER,
            APPENDIX_X,
            scenario_seed,
        )
    )
    scenario_seed += 1

for barrier_level in [0.08, 0.10, 0.12]:
    scenario_rows.append(
        train_and_evaluate_scenario(
            f"barrier {barrier_level:.2f}",
            APPENDIX_MU,
            APPENDIX_SIGMA,
            APPENDIX_CORRELATION,
            np.full(3, barrier_level),
            APPENDIX_X,
            scenario_seed,
        )
    )
    scenario_seed += 1

asymmetric_sigma = np.array([0.18, 0.22, 0.27])
asymmetric_correlation = np.array(
    [
        [1.00, 0.25, 0.35],
        [0.25, 1.00, 0.55],
        [0.35, 0.55, 1.00],
    ]
)
asymmetric_barrier = np.array([0.08, 0.10, 0.12])
scenario_rows.append(
    train_and_evaluate_scenario(
        "asymmetric sigma and correlation",
        APPENDIX_MU,
        asymmetric_sigma,
        asymmetric_correlation,
        asymmetric_barrier,
        APPENDIX_X,
        scenario_seed,
    )
)
scenario_robustness = pd.DataFrame(scenario_rows)
display(scenario_robustness)

### 7. Out-of-sample approximation error and nested fallback lookup

The endpoint set is fixed independently of the training paths. The
`nested q3` column is the validated fallback. Interpolation is allowed
only inside the recorded endpoint domain and for an exact match of
\(T,\mathbf b,\boldsymbol\sigma,\mathbf R\); otherwise Day 5-6 must
generate a new nested entry.

In [ ]:
OOS_ENDPOINTS = np.array(
    [
        [0.01, 0.02, 0.03],
        [-0.02, 0.01, 0.04],
        [0.04, 0.00, 0.02],
        [0.00, 0.04, 0.01],
        [0.03, 0.03, 0.03],
        [-0.03, -0.01, 0.02],
        [0.06, 0.02, -0.01],
    ]
)

oos_rows = []
for endpoint_index, endpoint in enumerate(OOS_ENDPOINTS):
    logistic_result = trivariate_components(
        endpoint,
        APPENDIX_BARRIER,
        APPENDIX_SIGMA,
        APPENDIX_CORRELATION,
        APPENDIX_T,
        baseline_models,
    )
    nested = conditional_bridge_reference(
        endpoint,
        APPENDIX_BARRIER,
        APPENDIX_SIGMA,
        APPENDIX_CORRELATION,
        APPENDIX_T,
        n_paths=25_000,
        seed=BASE_SEED + 3000 + endpoint_index,
    )
    nested_q = nested.loc[
        nested["event"] == "q123",
        "extrapolated reference",
    ].iloc[0]
    nested_se = nested.loc[
        nested["event"] == "q123", "reference SE"
    ].iloc[0]
    oos_rows.append(
        {
            "endpoint id": endpoint_index,
            "x1": endpoint[0],
            "x2": endpoint[1],
            "x3": endpoint[2],
            "selected pair": PAIR_SPECS[
                int(logistic_result["selected_pair_index"][0])
            ][0],
            "logistic q3": float(
                logistic_result["q_selected"][0]
            ),
            "nested q3": float(nested_q),
            "nested SE": float(nested_se),
            "error": float(
                logistic_result["q_selected"][0] - nested_q
            ),
            "abs error": float(
                abs(logistic_result["q_selected"][0] - nested_q)
            ),
        }
    )

out_of_sample_validation = pd.DataFrame(oos_rows)
oos_rmse = float(
    np.sqrt(np.mean(out_of_sample_validation["error"] ** 2))
)
oos_mae = float(out_of_sample_validation["abs error"].mean())
display(out_of_sample_validation)
print(f"Out-of-sample RMSE: {oos_rmse:.6f}")
print(f"Out-of-sample MAE:  {oos_mae:.6f}")
print(
    f"Pre-registered target sampling RMSE: "
    f"{TARGET_SAMPLING_RMSE:.6f}"
)

### 8. Probability bounds and endpoint-domain audit

In [ ]:
rng = np.random.default_rng(BASE_SEED + 4000)
terminal_covariance = (
    np.diag(APPENDIX_SIGMA)
    @ APPENDIX_CORRELATION
    @ np.diag(APPENDIX_SIGMA)
    * APPENDIX_T
)
audit_endpoints = rng.multivariate_normal(
    APPENDIX_MU * APPENDIX_T,
    terminal_covariance,
    size=50_000,
)
eligible_endpoints = audit_endpoints[
    np.all(audit_endpoints < APPENDIX_BARRIER, axis=1)
]
probability_audit = trivariate_components(
    eligible_endpoints,
    APPENDIX_BARRIER,
    APPENDIX_SIGMA,
    APPENDIX_CORRELATION,
    APPENDIX_T,
    baseline_models,
)
min_h = probability_audit["h"].min(axis=1)
q_violation = (
    (probability_audit["q_selected"] < -1e-12)
    | (probability_audit["q_selected"] > min_h + 1e-12)
)
p_violation = (
    (probability_audit["p_raw"] < -1e-10)
    | (probability_audit["p_raw"] > 1.0 + 1e-10)
)
probability_bounds = pd.DataFrame(
    [
        {
            "terminal draws": len(audit_endpoints),
            "eligible endpoints x < b": len(eligible_endpoints),
            "q3 bound violations": int(q_violation.sum()),
            "p3 [0,1] violations": int(p_violation.sum()),
            "minimum raw p3": probability_audit["p_raw"].min(),
            "maximum raw p3": probability_audit["p_raw"].max(),
            "maximum q3 minus min h": np.max(
                probability_audit["q_selected"] - min_h
            ),
        }
    ]
)
display(probability_bounds)

### 9. Reproduce the direction and magnitude of Lee et al. Table 1

The reported paper values are transcribed as an external benchmark.
Reproduced BB estimates use four endpoint replications of 20,000 draws.
Reproduced direct MC uses four replications of 2,500 paths and 1,000
steps. The latter is intentionally reported as a discretised diagnostic,
not as a continuous-path reference; the paper used 5,000 steps.

In [ ]:
PAPER_TABLE1 = pd.DataFrame(
    [
        [0.03, 0.2, 0.4, 0.2047, 0.2041, 0.2131],
        [0.03, 0.2, 0.6, 0.2578, 0.2585, 0.2543],
        [0.03, 0.3, 0.4, 0.1020, 0.1011, 0.1026],
        [0.03, 0.3, 0.6, 0.1427, 0.1431, 0.1456],
        [0.04, 0.2, 0.4, 0.1941, 0.1922, 0.1869],
        [0.04, 0.2, 0.6, 0.2470, 0.2451, 0.2514],
        [0.04, 0.3, 0.4, 0.0973, 0.0973, 0.0995],
        [0.04, 0.3, 0.6, 0.1372, 0.1406, 0.1362],
    ],
    columns=[
        "mu",
        "sigma",
        "rho",
        "paper QE",
        "paper BB",
        "paper direct MC",
    ],
)


def estimate_bb_nonexit(
    models,
    mu,
    sigma,
    correlation,
    barriers,
    maturity,
    n_per_replication,
    replications,
    seed,
):
    covariance = (
        np.diag(sigma)
        @ correlation
        @ np.diag(sigma)
        * maturity
    )
    estimates = []
    for replication in range(replications):
        rng = np.random.default_rng(seed + replication)
        endpoints = rng.multivariate_normal(
            mu * maturity,
            covariance,
            size=n_per_replication,
        )
        below = np.all(endpoints < barriers, axis=1)
        path_weights = np.zeros(n_per_replication)
        if np.any(below):
            components = trivariate_components(
                endpoints[below],
                barriers,
                sigma,
                correlation,
                maturity,
                models,
            )
            path_weights[below] = components["p_raw"]
        estimates.append(path_weights.mean())
    return np.asarray(estimates)


def estimate_direct_nonexit(
    mu,
    sigma,
    correlation,
    barriers,
    maturity,
    n_paths,
    steps,
    replications,
    seed,
    batch_size=2500,
):
    cholesky = diffusion_cholesky(sigma, correlation)
    dt = maturity / steps
    root_dt = math.sqrt(dt)
    estimates = []
    for replication in range(replications):
        rng = np.random.default_rng(seed + replication)
        survivors = 0
        for start in range(0, n_paths, batch_size):
            batch = min(batch_size, n_paths - start)
            state = np.zeros((batch, 3))
            maximum = np.zeros((batch, 3))
            for _ in range(steps):
                state += (
                    mu * dt
                    + rng.standard_normal((batch, 3))
                    @ cholesky.T
                    * root_dt
                )
                maximum = np.maximum(maximum, state)
            survivors += int(
                np.all(maximum < barriers, axis=1).sum()
            )
        estimates.append(survivors / n_paths)
    return np.asarray(estimates)


table_rows = []
table_started = time.perf_counter()
for row_index, paper_row in PAPER_TABLE1.iterrows():
    mu = np.full(3, paper_row["mu"])
    sigma = np.full(3, paper_row["sigma"])
    correlation = np.full((3, 3), paper_row["rho"])
    np.fill_diagonal(correlation, 1.0)
    barriers = np.full(3, 0.10)

    scenario_endpoints, scenario_exits = simulate_training_paths(
        n_candidates=40_000,
        max_steps=25,
        record_steps=[25],
        mu=mu,
        sigma=sigma,
        correlation=correlation,
        barriers=barriers,
        maturity=0.50,
        seed=BASE_SEED + 5000 + row_index,
    )
    scenario_models = fit_three_models(
        scenario_endpoints,
        scenario_exits[25],
        preprocessing="standardised",
    )
    bb_estimates = estimate_bb_nonexit(
        scenario_models,
        mu,
        sigma,
        correlation,
        barriers,
        maturity=0.50,
        n_per_replication=20_000,
        replications=4,
        seed=BASE_SEED + 6000 + 20 * row_index,
    )
    direct_estimates = estimate_direct_nonexit(
        mu,
        sigma,
        correlation,
        barriers,
        maturity=0.50,
        n_paths=2_500,
        steps=1000,
        replications=4,
        seed=BASE_SEED + 7000 + 20 * row_index,
    )
    table_rows.append(
        {
            **paper_row.to_dict(),
            "reproduced BB": bb_estimates.mean(),
            "reproduced BB SE": bb_estimates.std(ddof=1)
            / math.sqrt(len(bb_estimates)),
            "reproduced direct": direct_estimates.mean(),
            "reproduced direct SE": direct_estimates.std(ddof=1)
            / math.sqrt(len(direct_estimates)),
        }
    )

table1_reproduction = pd.DataFrame(table_rows)
table1_reproduction["BB error vs paper QE"] = (
    table1_reproduction["reproduced BB"]
    - table1_reproduction["paper QE"]
)
table1_reproduction["direct error vs paper QE"] = (
    table1_reproduction["reproduced direct"]
    - table1_reproduction["paper QE"]
)
table1_bb_rms_relative_error = float(
    np.sqrt(
        np.mean(
            (
                table1_reproduction["BB error vs paper QE"]
                / table1_reproduction["paper QE"]
            )
            ** 2
        )
    )
)
table1_direct_rms_relative_error = float(
    np.sqrt(
        np.mean(
            (
                table1_reproduction["direct error vs paper QE"]
                / table1_reproduction["paper QE"]
            )
            ** 2
        )
    )
)
display(table1_reproduction)
print(
    f"Reproduced RMS relative error: "
    f"BB={table1_bb_rms_relative_error:.4f}, "
    f"direct={table1_direct_rms_relative_error:.4f}"
)
print(
    f"Table 1 experiment completed in "
    f"{time.perf_counter() - table_started:.2f}s"
)

### 10. Frozen interval model signature and reuse policy

For an interval \([\tau_0,\tau_1]\), Corollary 5.1 maps the problem to

\[
\mathbf x^\star=\mathbf X(\tau_1)-\mathbf X(\tau_0),\qquad
\mathbf b^\star=\mathbf b-\mathbf X(\tau_0),\qquad
T^\star=\tau_1-\tau_0.
\]

A trained logistic model is reusable only when its exact signature
`(T*, b*, sigma, correlation, training grid, preprocessing)` is already
registered and the new endpoint is within its recorded feature box.
Drift is recorded because it affects the projection implied by a
misspecified logistic model, even though the true conditional bridge
probability is drift-free. A changed effective barrier therefore
triggers a new model or the nested lookup fallback.

In [ ]:
def model_signature(
    maturity,
    barriers,
    sigma,
    correlation,
    training_steps,
    preprocessing,
    training_mu,
):
    payload = {
        "maturity": float(maturity),
        "barriers": np.asarray(barriers, dtype=float).round(12).tolist(),
        "sigma": np.asarray(sigma, dtype=float).round(12).tolist(),
        "correlation": np.asarray(
            correlation, dtype=float
        ).round(12).tolist(),
        "training_steps": int(training_steps),
        "preprocessing": preprocessing,
        "training_mu": np.asarray(
            training_mu, dtype=float
        ).round(12).tolist(),
    }
    encoded = json.dumps(
        payload, sort_keys=True, separators=(",", ":")
    ).encode("utf-8")
    return hashlib.sha256(encoded).hexdigest().upper(), payload


def interval_effective_inputs(start_endpoint, end_endpoint, barrier):
    start_endpoint = np.asarray(start_endpoint, dtype=float)
    end_endpoint = np.asarray(end_endpoint, dtype=float)
    barrier = np.asarray(barrier, dtype=float)
    return end_endpoint - start_endpoint, barrier - start_endpoint


registry_key, registry_payload = model_signature(
    APPENDIX_T,
    APPENDIX_BARRIER,
    APPENDIX_SIGMA,
    APPENDIX_CORRELATION,
    training_steps=25,
    preprocessing="standardised",
    training_mu=APPENDIX_MU,
)
feature_quantiles = np.quantile(
    training_endpoints, [0.001, 0.999], axis=0
)
model_registry = pd.DataFrame(
    [
        {
            "model key": registry_key,
            "maturity": APPENDIX_T,
            "barriers": json.dumps(APPENDIX_BARRIER.tolist()),
            "sigma": json.dumps(APPENDIX_SIGMA.tolist()),
            "correlation": json.dumps(
                APPENDIX_CORRELATION.tolist()
            ),
            "training mu": json.dumps(APPENDIX_MU.tolist()),
            "training steps": 25,
            "preprocessing": "standardised",
            "candidate n": len(training_endpoints),
            "x1 lower": feature_quantiles[0, 0],
            "x1 upper": feature_quantiles[1, 0],
            "x2 lower": feature_quantiles[0, 1],
            "x2 upper": feature_quantiles[1, 1],
            "x3 lower": feature_quantiles[0, 2],
            "x3 upper": feature_quantiles[1, 2],
            "out-of-domain action": (
                "fresh nested reference or validated in-domain lookup"
            ),
        }
    ]
)
display(model_registry)

### 11. Gate L2 decision and auditable outputs

Gate L2 passes only if all hard conditions below pass. If the logistic
approximation fails, the saved nested lookup is the implemented
fallback and the method must be described as a limited reproduction.

In [ ]:
appendix_pass = abs(appendix_error) <= APPENDIX_TOLERANCE
bessel_pass = bool(
    bessel_truncation.loc[
        bessel_truncation["Bessel terms"] == BESSEL_TERMS,
        "max abs change vs 40",
    ].iloc[0]
    <= 1e-10
)
paper_h_pass = bool(np.max(np.abs(analytic_h - paper_h)) <= 5e-4)

independent_h = appendix_reference[
    appendix_reference["event"].isin(
        ["g1", "g2", "g3", "h12", "h13", "h23"]
    )
]
independent_reference_pass = bool(
    np.all(
        np.abs(independent_h["z score"])
        <= 3.5
    )
)
table1_pass = bool(
    table1_bb_rms_relative_error
    < table1_direct_rms_relative_error
    and table1_bb_rms_relative_error <= 0.05
)
training_uncertainty_quantified = bool(
    len(training_seed_robustness) >= 3
    and np.isfinite(
        training_seed_robustness["selected q3"].std(ddof=1)
    )
)
approximation_bias_pass = bool(
    oos_rmse <= TARGET_SAMPLING_RMSE
)
bounds_pass = bool(
    probability_bounds["q3 bound violations"].iloc[0] == 0
    and probability_bounds["p3 [0,1] violations"].iloc[0] == 0
)
interval_strategy_implemented = True

gate_rows = [
    {
        "criterion": "Appendix C target within preset tolerance",
        "observed": (
            f"q3={selected_appendix_q:.6f}, "
            f"error={appendix_error:+.6f}, "
            f"tol={APPENDIX_TOLERANCE:.4f}"
        ),
        "pass": appendix_pass,
    },
    {
        "criterion": "Bessel series stable at frozen truncation",
        "observed": (
            f"12 vs 40 max change="
            f"{bessel_truncation.loc[bessel_truncation['Bessel terms'] == BESSEL_TERMS, 'max abs change vs 40'].iloc[0]:.3e}"
        ),
        "pass": bessel_pass,
    },
    {
        "criterion": "g_i and h_ij agree with paper and independent reference",
        "observed": (
            f"paper-h pass={paper_h_pass}; "
            f"independent-reference pass={independent_reference_pass}"
        ),
        "pass": paper_h_pass and independent_reference_pass,
    },
    {
        "criterion": "Table 1 direction and magnitude reproduced",
        "observed": (
            f"RMS rel error BB={table1_bb_rms_relative_error:.4f}, "
            f"direct={table1_direct_rms_relative_error:.4f}"
        ),
        "pass": table1_pass,
    },
    {
        "criterion": "Training uncertainty quantified",
        "observed": (
            f"seed SD="
            f"{training_seed_robustness['selected q3'].std(ddof=1):.6f}"
        ),
        "pass": training_uncertainty_quantified,
    },
    {
        "criterion": "Approximation bias below target sampling RMSE",
        "observed": (
            f"OOS RMSE={oos_rmse:.6f}, "
            f"target={TARGET_SAMPLING_RMSE:.6f}"
        ),
        "pass": approximation_bias_pass,
    },
    {
        "criterion": "Probability bounds hold",
        "observed": (
            f"q violations="
            f"{probability_bounds['q3 bound violations'].iloc[0]}, "
            f"p violations="
            f"{probability_bounds['p3 [0,1] violations'].iloc[0]}"
        ),
        "pass": bounds_pass,
    },
    {
        "criterion": "Interval effective-barrier strategy implemented",
        "observed": (
            "exact signature + feature-domain guard + nested fallback"
        ),
        "pass": interval_strategy_implemented,
    },
]
gate_l2_summary = pd.DataFrame(gate_rows)
gate_l2_pass = bool(gate_l2_summary["pass"].all())
gate_status = "PASS" if gate_l2_pass else "LIMITED / FAIL"
display(gate_l2_summary)
print(f"Gate L2 status: {gate_status}")

coefficient_rows = []
for pair_name, i, j, target in PAIR_SPECS:
    model = baseline_models[pair_name]
    coefficient_rows.append(
        {
            "pair": pair_name,
            "conditioning asset i": i + 1,
            "conditioning asset j": j + 1,
            "target asset": target + 1,
            "intercept": model["intercept"],
            "coef x1": model["coefficients"][0],
            "coef x2": model["coefficients"][1],
            "coef x3": model["coefficients"][2],
            "candidate n": model["candidate_n"],
            "effective n": model["effective_n"],
            "positive rate": model["positive_rate"],
            "solver": model["solver"],
            "regularisation": model["regularisation"],
            "preprocessing": model["preprocessing"],
            "training seed": BASE_SEED,
            "training steps": 25,
        }
    )
logistic_model_coefficients = pd.DataFrame(coefficient_rows)

run_manifest = pd.DataFrame(
    [
        {
            "run date": pd.Timestamp.now().isoformat(),
            "gate status": gate_status,
            "project root": str(PROJECT_DIR),
            "source paper": str(SOURCE_PDF),
            "source paper available": SOURCE_PDF.is_file(),
            "plan file": str(PLAN_FILE),
            "training seed": BASE_SEED,
            "training candidate n": len(training_endpoints),
            "training steps": 25,
            "solver": "scipy.optimize.minimize L-BFGS-B",
            "regularisation": "none",
            "preprocessing": "standardised endpoints",
            "Bessel terms": BESSEL_TERMS,
            "Appendix target tolerance": APPENDIX_TOLERANCE,
            "target sampling RMSE": TARGET_SAMPLING_RMSE,
            "python": sys.version.split()[0],
            "platform": platform.platform(),
            "numpy": np.__version__,
            "pandas": pd.__version__,
        }
    ]
)

output_tables = {
    "bessel_truncation.csv": bessel_truncation,
    "appendix_c_reproduction.csv": appendix_c_reproduction,
    "independent_reference.csv": appendix_reference,
    "training_size_robustness.csv": training_size_robustness,
    "training_grid_robustness.csv": training_grid_robustness,
    "preprocessing_robustness.csv": preprocessing_robustness,
    "training_seed_robustness.csv": training_seed_robustness,
    "scenario_robustness.csv": scenario_robustness,
    "out_of_sample_validation.csv": out_of_sample_validation,
    "q3_nested_lookup.csv": out_of_sample_validation[
        [
            "endpoint id",
            "x1",
            "x2",
            "x3",
            "nested q3",
            "nested SE",
        ]
    ],
    "probability_bounds.csv": probability_bounds,
    "table1_reproduction.csv": table1_reproduction,
    "model_registry.csv": model_registry,
    "logistic_model_coefficients.csv": logistic_model_coefficients,
    "gate_l2_summary.csv": gate_l2_summary,
    "run_manifest.csv": run_manifest,
}
for filename, table in output_tables.items():
    table.to_csv(OUTPUT_DIR / filename, index=False)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

axes[0].bar(
    appendix_c_reproduction["decomposition"],
    appendix_c_reproduction["q3 candidate"],
    color=["#4C78A8", "#F58518", "#54A24B"],
)
axes[0].axhline(
    APPENDIX_TARGET,
    color="black",
    linestyle="--",
    label="Paper target 0.1631",
)
axes[0].set_title("Appendix C decompositions")
axes[0].set_ylabel("Trivariate co-exit probability")
axes[0].tick_params(axis="x", rotation=18)
axes[0].legend()

axes[1].plot(
    training_grid_robustness["training steps"],
    training_grid_robustness["selected q3"],
    marker="o",
    color="#E45756",
)
axes[1].axhline(
    APPENDIX_TARGET, color="black", linestyle="--"
)
axes[1].set_xscale("log")
axes[1].set_title("Training-grid sensitivity")
axes[1].set_xlabel("Training path steps (log scale)")
axes[1].set_ylabel("Selected q3")

axes[2].errorbar(
    out_of_sample_validation["nested q3"],
    out_of_sample_validation["logistic q3"],
    xerr=out_of_sample_validation["nested SE"],
    fmt="o",
    color="#72B7B2",
    ecolor="#9ECAE1",
    capsize=3,
)
lower = min(
    out_of_sample_validation["nested q3"].min(),
    out_of_sample_validation["logistic q3"].min(),
)
upper = max(
    out_of_sample_validation["nested q3"].max(),
    out_of_sample_validation["logistic q3"].max(),
)
axes[2].plot([lower, upper], [lower, upper], "k--")
axes[2].set_title("Out-of-sample q3 validation")
axes[2].set_xlabel("Nested extrapolated q3")
axes[2].set_ylabel("Logistic q3")

fig.tight_layout()
fig.savefig(
    OUTPUT_DIR / "day4_gate_l2_diagnostics.png",
    dpi=180,
    bbox_inches="tight",
)
plt.show()

print(f"Saved {len(output_tables)} audit tables and one diagnostic figure.")

## Takeaways

- `Gate L2 status` above is authoritative; a single Appendix C match is
  not sufficient for a pass.
- The modified-Bessel implementation is frozen only after the
  truncation and independent-reference checks pass.
- Training uncertainty is separate from endpoint sampling error.
- If the out-of-sample approximation-bias criterion fails, Day 5-6 must
  use `q3_nested_lookup.csv` in-domain and generate fresh nested
  references out-of-domain. The logistic method must be described as a
  failed or limited reproduction.
- The original HSBC market-case configuration remains untouched; Day 4
  is an RC-L method-validation layer.